# Retail Dashboard (Jupyter — no Streamlit)

**Run each cell in order (Shift+Enter).** All graphs and metrics appear directly in the notebook.

Make sure **Online Retail.xlsx** is in the same folder as this notebook.

## 1. Load data and compute RFM

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import timedelta
import plotly.express as px

# Folder with this notebook and Online Retail.xlsx
BASE = Path.cwd()
DATA_FILE = BASE / "Online Retail.xlsx"
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Put 'Online Retail.xlsx' in: {BASE}")

df = pd.read_excel(DATA_FILE, engine="openpyxl")
df = df.drop_duplicates()
req = ["CustomerID", "Description", "InvoiceNo", "StockCode", "InvoiceDate", "Quantity", "UnitPrice", "Country"]
df = df.dropna(subset=[c for c in req if c in df.columns])
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
df["CustomerID"] = df["CustomerID"].astype(int)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
clean = df.copy()

ref = clean["InvoiceDate"].max() + timedelta(days=1)
rfm = clean.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (ref - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum"),
).reset_index()
rfm["R_Score"] = pd.qcut(rfm["Recency"], q=4, labels=[4,3,2,1], duplicates="drop")
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), q=4, labels=[1,2,3,4], duplicates="drop")
rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"), q=4, labels=[1,2,3,4], duplicates="drop")
rfm["R_Score"] = rfm["R_Score"].astype(int)
rfm["F_Score"] = rfm["F_Score"].astype(int)
rfm["M_Score"] = rfm["M_Score"].astype(int)

def seg(row):
    r,f,m = row["R_Score"], row["F_Score"], row["M_Score"]
    if r>=4 and f>=3 and m>=3: return "Champions"
    if r>=3 and f>=2 and m>=2: return "Loyal"
    if r>=3 and (f<=2 or m<=2): return "Potential loyal"
    if r==2 and f>=2 and m>=2: return "At risk"
    if r<=2 and f>=3 and m>=3: return "Can't lose"
    if r<=2 and f<=2 and m>=2: return "Hibernating"
    if r<=2 and f<=2 and m<=2: return "Lost"
    if r>=3 and f<=1 and m<=1: return "New"
    if r<=1: return "Lost"
    return "Other"
rfm["Segment"] = rfm.apply(seg, axis=1)

print("Data loaded.")
print(f"Rows: {len(clean):,}  |  Customers: {rfm['CustomerID'].nunique():,}")

## 2. Business Overview — KPIs

In [ ]:
total_rev = clean["Revenue"].sum()
n_cust = clean["CustomerID"].nunique()
n_orders = clean["InvoiceNo"].nunique()
aov = clean.groupby("InvoiceNo")["Revenue"].sum().mean()
raw = pd.read_excel(DATA_FILE, engine="openpyxl")
inv = raw["InvoiceNo"].astype(str)
ret_rate = raw.loc[inv.str.startswith("C"), "InvoiceNo"].nunique() / raw["InvoiceNo"].nunique() * 100

print(f"Total Revenue:     £{total_rev/1e6:.2f}M")
print(f"Customers:         {n_cust:,}")
print(f"Orders:            {n_orders:,}")
print(f"AOV:               £{aov:,.2f}")
print(f"Return/Cancel rate: {ret_rate:.1f}%")

## 3. Revenue trend (monthly)

In [ ]:
clean["YearMonth"] = clean["InvoiceDate"].dt.to_period("M").astype(str)
trends = clean.groupby("YearMonth").agg(Revenue=("Revenue", "sum"), Orders=("InvoiceNo", "nunique")).reset_index()
fig = px.line(trends, x="YearMonth", y="Revenue", title="Revenue trend (monthly)", markers=True)
fig.update_layout(yaxis_tickformat="£,.0f", xaxis_tickangle=-45)
fig.show()

## 4. Revenue by country (top 15)

In [ ]:
ov = clean.groupby("InvoiceNo").agg(Rev=("Revenue", "sum")).reset_index()
ov = ov.merge(clean[["InvoiceNo", "Country"]].drop_duplicates(), on="InvoiceNo")
geo = ov.groupby("Country")["Rev"].sum().reset_index(name="Revenue").sort_values("Revenue", ascending=False).head(15)
fig = px.bar(geo, x="Country", y="Revenue", title="Revenue by country (top 15)", color="Revenue", color_continuous_scale="Blues")
fig.update_layout(showlegend=False, xaxis_tickangle=-45, yaxis_tickformat="£,.0f")
fig.show()

## 5. Top 15 products by revenue

In [ ]:
top = clean.groupby(["StockCode", "Description"]).agg(Revenue=("Revenue", "sum")).reset_index().nlargest(15, "Revenue")
top["Desc"] = top["Description"].fillna("").astype(str).str[:40]
fig = px.bar(top, x="Revenue", y="Desc", orientation="h", title="Top 15 products by revenue", color="Revenue", color_continuous_scale="Teal")
fig.update_layout(showlegend=False, xaxis_tickformat="£,.0f", yaxis=dict(autorange="reversed"))
fig.show()

---
## 6. Customer RFM lookup

**Change the Customer ID below and re-run the cell** to see that customer's segment, recommendation, and charts.

In [ ]:
CUSTOMER_ID = 12347   # <-- Change this and re-run to see another customer

cust = rfm[rfm["CustomerID"] == CUSTOMER_ID]
if cust.empty:
    print(f"Customer {CUSTOMER_ID} not found. Try another ID.")
else:
    cust = cust.iloc[0]
    seg = cust["Segment"]
    rec = {
        "Champions": "Retain & reward — VIP programme, early access, loyalty offers.",
        "Loyal": "Retain — keep engaged to avoid slipping to At risk.",
        "At risk": "Win-back urgently — personalised offer, we miss you campaign.",
        "Can't lose": "Win-back urgently — high value; personal outreach, strong incentive.",
        "Hibernating": "Re-engage — win-back email, targeted offer.",
        "Potential loyal": "Nurture — second-purchase offer, recommendations.",
        "Lost": "Low-cost re-engage — simple win-back; avoid heavy discount.",
        "New": "Onboard — welcome series, first repeat incentive.",
        "Other": "Review — assess next best action.",
    }.get(seg, "Review segment and define action.")
    print(f"Customer {CUSTOMER_ID}  |  Segment: {seg}")
    print(f"Recommendation: {rec}")
    
    co = clean[clean["CustomerID"] == CUSTOMER_ID]
    if not co.empty:
        print(f"Country: {co['Country'].iloc[0]}  |  First purchase: {co['InvoiceDate'].min().date()}  |  Last: {co['InvoiceDate'].max().date()}")
        tot = co["Revenue"].sum()
        ntx = co["InvoiceNo"].nunique()
        print(f"Total revenue: £{tot:,.2f}  |  AOV: £{tot/ntx:,.2f}  |  Transactions: {ntx}")

## 7. Top products for this customer

In [ ]:
co = clean[clean["CustomerID"] == CUSTOMER_ID]
if co.empty:
    print("No data for this customer.")
else:
    prod = co.groupby("Description").agg(Revenue=("Revenue", "sum")).reset_index().sort_values("Revenue", ascending=False).head(12)
    prod["Desc"] = prod["Description"].fillna("").astype(str).str[:40]
    fig = px.bar(prod, x="Revenue", y="Desc", orientation="h", color="Revenue", color_continuous_scale="Viridis")
    fig.update_layout(showlegend=False, xaxis_tickformat="£,.0f", yaxis=dict(autorange="reversed"), height=400)
    fig.show()

## 8. Spend by product category (first word)

In [ ]:
co = clean[clean["CustomerID"] == CUSTOMER_ID].copy()
if co.empty:
    print("No data for this customer.")
else:
    co["Category"] = co["Description"].str.split().str[0].fillna("Other")
    cat = co.groupby("Category")["Revenue"].sum().reset_index().sort_values("Revenue", ascending=False).head(10)
    fig = px.pie(cat, values="Revenue", names="Category", color_discrete_sequence=px.colors.qualitative.Set3)
    fig.update_traces(textposition="inside", textinfo="percent+label")
    fig.show()